In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage,HumanMessage, BaseMessage
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

c:\Users\hshinde3\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

In [5]:
llm.invoke("Testing if pinging to llm. Only confirm yes.").content

'Yes'

In [4]:
### creating state
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [5]:
def chat_node(state:ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages':[response]}
    

In [6]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)


In [7]:
chatbot.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| chat_node |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [9]:
initial_state = [HumanMessage(content="What is the capital of India?")]

In [12]:
for message_chunk, metadata in chatbot.stream(
    {'messages':[HumanMessage(content="Tell me a popular indian recipe in less than 500 words.")]}, 
    config = {'configurable':{'thread_id':'thread-1'}},
    stream_mode = "messages"
    ):
    if message_chunk.content:
        print(message_chunk.content, end=" ", flush=True)


**Butter Chicken (Murgh Makhani)**

Butter Chicken is perhaps the most iconic Indian dish globally.  Originating in Delhi in the 1950s, it features tender chicken in a velvety, mildly spiced tomato and cream  sauce.

### **Ingredients**

**For the Marinade:**
*   500g boneless chicken  (thighs are best), bite-sized pieces
*   1/2 cup plain yogurt
*   1 tbsp  ginger-garlic paste
*   1 tsp Kashmiri red chili powder (or paprika)
*   1 tsp garam masala
*    Salt to taste

**For the Gravy:**
*   2 tbsp butter + 1 tbsp oil
*   1  cup tomato purée (fresh or canned)
*   1 tbsp sugar or honey
*   1 tsp kasuri methi (dried  fenugreek leaves), crushed
*   1/2 cup heavy cream
*   1 tsp garam masala

###  **Instructions**

1.  **Marinate:** In a bowl, mix the chicken with the yogurt, ginger-garlic  paste, chili powder, garam masala, and salt. Let it rest for at least 30 minutes (ideally  3 hours).
2.  **Cook the Chicken:** Heat oil in a large pan over medium-high heat. Add  the chicken pieces in ba

In [11]:
type(stream)

generator